# Contratos y trazabilidad de datos

Bronze Snowflake → staging/intermediate dbt → OBT SYSTEM → snapshot PostgreSQL. El modelo Kimball de PowerBI se conserva.

Las funciones de producción son la única implementación del entrenamiento. Estos experimentos registran artefactos en MLflow; no sustituyen la publicación activa. El DAG publica los siete módulos de forma atómica. No se consultan datos personales de contacto.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from sqlalchemy import text
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'hgc-ml').exists())
sys.path.insert(0, str(ROOT / 'hgc-ml'))
from core.settings import postgres_engine
from training.common import setup_tracking
engine = postgres_engine()
setup_tracking()
with engine.connect() as connection:
    load_id = connection.execute(text('SELECT load_id FROM hgc_analytics.loads ORDER BY loaded_at DESC LIMIT 1')).scalar_one()
print('Snapshot:', load_id)


Snapshot: befab29d-54d0-4c88-aee6-0d59c6e120aa


In [2]:
from data.contracts import MODELS, validate
summary=[]
with engine.connect().execution_options(isolation_level='REPEATABLE READ') as connection:
    with connection.begin():
        for name, (_, keys) in MODELS.items():
            frame=pd.read_sql_table(name,connection,schema='hgc_analytics')
            validate(frame,keys)
            summary.append({'obt':name,'rows':len(frame),'grain':', '.join(keys),'valid':True})
            del frame
pd.DataFrame(summary)


,obt,rows,grain,valid
0,sys_sales_daily,29650,"id_sucursal, fecha",True
1,sys_customer_features,943568,"id_cliente, cutoff",True
2,sys_customer_orders,750578,id_pedido,True
3,sys_branch_monthly,977,"id_sucursal, mes_fecha",True
4,sys_product_weekly,38052,"id_sucursal, id_producto, semana",True


In [3]:
with engine.connect() as connection:
    cohorts=pd.read_sql_query(text('SELECT cutoff, COUNT(*) AS customers, COUNT(spend_90d) AS mature_labels FROM hgc_analytics.sys_customer_features GROUP BY cutoff ORDER BY cutoff'),connection)
assert cohorts.iloc[-1].mature_labels == 0
assert (cohorts.iloc[:-1].mature_labels > 0).all()
cohorts


,cutoff,customers,mature_labels
0,2025-01-12,119619,119619
1,2025-04-12,118389,118389
2,2025-07-11,117744,117744
3,2025-10-09,116378,116378
4,2026-01-07,116302,116302
5,2026-04-07,117979,117979
6,2026-07-06,118659,118659
7,2026-10-04,118498,0
